# Scratch Work for Flow Matching model 

In [1]:
from ase.io import read
from ase import Atoms
from ase.visualize import view

import numpy as np

import torch 
from torch import optim
from torch import nn
from torch_geometric.data import Data
from torch import Tensor

from collections import deque

from adsflow.models.gemnet.gemnet_flow import GemNetFlow
from adsflow.models.gemnet.gemnet_base import GemNetOC
from ase.io import read


/opt/anaconda3/envs/cfm/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Compile GemNet for flow matching

In [2]:
# GemNet-OC params
model_cfg = {
    "num_targets": 1,
    "num_spherical": 7,
    "num_radial": 128,
    "num_blocks": 4,
    "emb_size_atom": 256,
    "emb_size_edge": 512,
    "emb_size_trip_in": 64,
    "emb_size_trip_out": 64,
    "emb_size_quad_in": 32,
    "emb_size_quad_out": 32,
    "emb_size_aint_in": 64,
    "emb_size_aint_out": 64,
    "emb_size_rbf": 16,
    "emb_size_cbf": 16,
    "emb_size_sbf": 32,
    "num_before_skip": 2,
    "num_after_skip": 2,
    "num_concat": 1,
    "num_atom": 3,
    "num_output_afteratom": 3,
    "num_atom_emb_layers": 0,
    "num_global_out_layers": 2,
    "flow_matching": True,
    "dummy": True,
    "use_pbc": True,
    "scale_backprop_forces": False,
    "cutoff": 12.0,
    "cutoff_qint": 12.0,
    "cutoff_aeaint": 12.0,
    "cutoff_aint": 12.0,
    "max_neighbors": 30,
    "max_neighbors_qint": 8,
    "max_neighbors_aeaint": 20,
    "max_neighbors_aint": 1000,
    "enforce_max_neighbors_strictly": True,
    "rbf": {"name": "gaussian"},
    "rbf_spherical": None,
    "envelope": {"name": "polynomial", "exponent": 5},
    "cbf": {"name": "spherical_harmonics"},
    "sbf": {"name": "legendre_outer"},
    "extensive": True,
    "forces_coupled": True,
    "output_init": "HeOrthogonal",
    "activation": "silu",
    "scale_file": None,
    "quad_interaction": True,
    "atom_edge_interaction": True,
    "edge_atom_interaction": True,
    "atom_interaction": True,
    "scale_basis": True,
}


In [3]:
GemNet = GemNetOC(**model_cfg)

# Count total number of parameters
trainable_params = sum(p.numel() for p in GemNet.parameters() if p.requires_grad)
print(f"Trainable parameters: {trainable_params}")

Trainable parameters: 37815808


## Testing GemNet-OC for energy/force regression

In [4]:
optimizer = optim.Adam(GemNet.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

# Training loop
def train():
    '''
    Defines the training procedure for the neural network
    '''
    l = []
    for epoch in range(1000): 
  
        # Create batch index (all atoms belong to structure 0)
        x1 = positions 
        batch = torch.zeros(len(x1), dtype=torch.long)

        # Build data objects
        data = Data(pos=x1, atomic_numbers=atomic_numbers, batch=batch)
        data.cell = torch.tensor(contcar.get_cell().array, dtype=torch.float32).unsqueeze(0) 
        data.pbc = torch.tensor([[True, True, True]])
        data.natoms = torch.tensor([len(positions)], dtype=torch.long)
        data.tags = torch.zeros(len(positions), dtype=torch.long)

        # Compute u_theta 
        output = GemNet(data)

        # Compute loss 
        force = torch.tensor([[0, 0, 1], [0, 0, -1]],  dtype=torch.float32)
        loss = loss_fn(output, force)
        l.append(loss.item())
        l_avg = np.mean(np.array(l[-50:]))
        print(f"Loss at epoch {epoch}: {loss.item()}")

        if loss.item() < 0.0000001:
            break 

        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()


    return output


# AdsFlow Training Loop

In [5]:
def train(
    model,
    positions,
    atomic_numbers,
    cell,
    optimizer,
    loss_fn,
    num_epochs: int = 100_000_000,
    print_every: int = 10,
    tol: float = 1e-2,
    device: torch.device = torch.device("cpu"),
):
    
    model.train()
    # Move static tensors once
    x1 = positions.to(device)
    atomic_numbers = atomic_numbers.to(device)
    cell = cell.to(device)
    batch = torch.zeros(len(x1), dtype=torch.long, device=device)

    # Keep a fixed‐size buffer for running avg
    recent_losses = deque(maxlen=print_every)

    for epoch in range(num_epochs):
        # build data object 
        data = Data(
            atomic_numbers=atomic_numbers,    
            batch=batch,
            cell = cell, 
            pbc = torch.tensor([[True, True, True]]), 
            natoms = torch.tensor([len(positions)], dtype=torch.long), 
            tags = torch.zeros(len(positions), dtype=torch.long)
        )
        # interpolate between noise and sample: optimal transport
        x0 = torch.randn_like(x1, device=device)
        t  = torch.rand(1, device=device)
        xt = (1 - t) * x0 + t * x1
        data.t = t
        data.pos = xt

        optimizer.zero_grad()
        u_theta = model(data)
        target  = x1 - x0
        loss    = loss_fn(u_theta, target)
        loss.backward()
        optimizer.step()

        recent_losses.append(loss.item())
        if epoch % print_every == 0:
            avg = sum(recent_losses) / len(recent_losses)
            print(f"[{epoch:6d}] loss = {loss.item()}, avg({print_every}) = {avg}")

        if loss.item() < tol:
            print(f"Converged at epoch {epoch} (loss {loss.item()})")
            break

    return 



In [8]:
class BaselineVel(nn.Module):
    def __init__(self):
        super().__init__()
        # Input dim = flattened x_t (2 atoms × 3 dims) + t (1 dim) = 7
        self.net = nn.Sequential(
            nn.Linear(4 * 3 + 1, 64),
            nn.ReLU(),
            nn.Linear(64, 4 * 3),   # output = 2 atoms × 3 dims
        )

    def forward(self, data):
        x_flat = data.pos.view(1, -1)  # [1,6]
        t_flat = data.t.view(1, 1)     # [1,1]
        out = self.net(torch.cat([x_flat, t_flat], dim=1))
        return out.view(-1, 3)         # [2,3]

In [9]:
contcar = read("POSCAR.vasp")
contcar = read("no3.xyz")

x1 = torch.tensor(contcar.get_positions(), dtype=torch.float32)
z = torch.tensor(contcar.get_atomic_numbers(), dtype=torch.long)
cell = torch.tensor(contcar.get_cell().array, dtype=torch.float32).unsqueeze(0)
model = BaselineVel() #

train(model = model, 
      positions = x1,
      atomic_numbers = z, 
      cell = cell, 
      optimizer = optim.Adam(model.parameters(), 1e-3), 
      loss_fn = nn.MSELoss(), 
      tol = 1e-3, 
      print_every = 100
      )

[     0] loss = 31.842180252075195, avg(100) = 31.842180252075195
[   100] loss = 7.800827503204346, avg(100) = 18.458630142211913
[   200] loss = 2.792996644973755, avg(100) = 8.810693410336972
[   300] loss = 1.1684190034866333, avg(100) = 7.559083667993545
[   400] loss = 7.946653366088867, avg(100) = 6.526249922513962
[   500] loss = 4.491513729095459, avg(100) = 4.7556550142169
[   600] loss = 16.187206268310547, avg(100) = 4.411552429199219
[   700] loss = 3.6979570388793945, avg(100) = 2.643196560740471
[   800] loss = 6.479654788970947, avg(100) = 2.7275887130200864
[   900] loss = 1.1502151489257812, avg(100) = 1.814930285513401
[  1000] loss = 0.4937514364719391, avg(100) = 1.3141285601258277
[  1100] loss = 2.088146448135376, avg(100) = 0.9997226457297802
[  1200] loss = 0.8503311276435852, avg(100) = 0.9096475456655025
[  1300] loss = 1.8237346410751343, avg(100) = 0.82043441593647
[  1400] loss = 0.6758060455322266, avg(100) = 0.716569904834032
[  1500] loss = 0.3091239035

## Sampler

In [13]:
# Sampler function

def step(model, xt: Tensor, t_start: Tensor, t_end: Tensor) -> Tensor: 

    # Define data object 
    data = Data() 
    data.atomic_numbers = z
    data.batch = torch.zeros(len(x1), dtype=torch.long)
    data.cell = torch.tensor(contcar.get_cell().array, dtype=torch.float32).unsqueeze(0) 
    data.pbc = torch.tensor([[True, True, True]])
    data.natoms = torch.tensor([len(xt)], dtype=torch.long)
    data.tags = torch.zeros(len(xt), dtype=torch.long)


    #t_start = t_start.view(1, 1).expand(xt.shape[0], 1)

    midpoint_time = t_start + (t_end - t_start) / 2 # evalute the midpoint 

    data.t = t_start
    data.pos = xt
    intermediate_value = model(data) # compute the velocity at this point using the trained neural network
    updated_xt = xt + intermediate_value * (t_end - t_start) / 2 # move xt using computed velocity 

    data.t = midpoint_time
    data.pos = updated_xt
    midpoint_output = model(data) # compute vt again 
    
    return xt + (t_end - t_start) * midpoint_output # compute xt

sample = True
if sample: 

    model.eval()  # Disables dropout/batch norm and ensures no gradients
    x0 = torch.randn_like(x1) 
    x = step(model, xt=x0, t_start=torch.Tensor([0]), t_end=torch.Tensor([1]))

    print(x0)
    print(x)
    
    # Sample from p1
    gen_struc = Atoms("NO3", positions=x.detach().numpy())

    view(gen_struc) 

tensor([[ 1.0314, -0.2807, -0.3247],
        [-0.4107,  0.4721, -1.0948],
        [ 1.1682, -2.4333, -0.1740],
        [-0.7055, -0.1691,  0.3482]])
tensor([[4.1470, 1.9491, 9.2210],
        [3.7108, 2.3529, 9.8768],
        [5.1499, 2.4877, 8.4991],
        [2.8939, 2.6763, 8.5142]], grad_fn=<AddBackward0>)
